In [ ]:
# FILE: generate_data.py
# Trial-to-Paid Conversion Optimizer — Synthetic Data Generator 
# Researched across companies for 7-day trial, 14-day trial etc user's data and asked for a pyhton script
# using prompt engineering and gen AI for a synthetic data that mimics real world users data for companies like
# Salesforce, amazon, Hubspot, postman .
# ─────────────────────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

np.random.seed(42)
random.seed(42)

# ── CONFIG ────────────────────────────────────────────────────────────────
N_USERS = 5000
START_DATE = datetime(2025, 1, 1)
END_DATE   = datetime(2025, 12, 31)
CONVERSION_RATE_CTRL = 0.08
CONVERSION_RATE_TRTM = 0.15

industries   = ['Technology', 'Finance', 'Healthcare', 'Retail', 'Education', 'Marketing']
company_sz   = ['1-10', '11-50', '51-200', '201-500', '500+']
plans        = ['Starter', 'Pro', 'Enterprise']
plan_mrr     = {'Starter': 29, 'Pro': 79, 'Enterprise': 199}
event_types  = ['login', 'feature_used', 'invite_sent', 'dashboard_created', 'support_ticket']

def rand_date(start, end):
    delta = end - start
    return start + timedelta(days=random.randint(0, delta.days))

# ── TABLE 1: users ────────────────────────────────────────────────────────
users = pd.DataFrame({
    'user_id': range(1001, 1001 + N_USERS),
    'signup_date': [rand_date(START_DATE, END_DATE).strftime('%Y-%m-%d')
                    for _ in range(N_USERS)],
    'industry': np.random.choice(industries, N_USERS),
    'company_size': np.random.choice(company_sz, N_USERS,
                        p=[0.35, 0.30, 0.20, 0.10, 0.05]),
    'onboarding_variant': np.random.choice(['Control', 'Treatment'], N_USERS)
})
users.to_csv('users.csv', index=False)

# ── TABLE 2: trials ───────────────────────────────────────────────────────
trial_rows = []
for _, u in users.iterrows():
    rate = (CONVERSION_RATE_TRTM if u['onboarding_variant'] == 'Treatment'
            else CONVERSION_RATE_CTRL)
    converted = int(np.random.rand() < rate)
    start = datetime.strptime(u['signup_date'], '%Y-%m-%d')
    end   = start + timedelta(days=14)
    conv_date = None
    if converted:
        conv_date = (start + timedelta(days=random.randint(3, 13))).strftime('%Y-%m-%d')
    trial_rows.append({
        'trial_id': 5000 + len(trial_rows) + 1,
        'user_id': u['user_id'],
        'trial_end_date': end.strftime('%Y-%m-%d'),
        'converted': converted,
        'conversion_date': conv_date
    })
trials = pd.DataFrame(trial_rows)
trials.to_csv('trials.csv', index=False)

# ── TABLE 3: events ───────────────────────────────────────────────────────
event_rows = []
converted_set = set(trials[trials['converted']==1]['user_id'])
eid = 10001
for _, u in users.iterrows():
    uid = u['user_id']
    is_conv = uid in converted_set
    start = datetime.strptime(u['signup_date'], '%Y-%m-%d')
    n_days = random.randint(8, 14) if is_conv else random.randint(1, 5)
    for d in range(n_days):
        edate = (start + timedelta(days=d)).strftime('%Y-%m-%d')
        for etype in np.random.choice(event_types,
                        size=random.randint(1,3), replace=False):
            event_rows.append({
                'event_id': eid, 'user_id': uid,
                'event_date': edate, 'event_type': etype,
                'event_count': random.randint(1, 10 if is_conv else 4)
            })
            eid += 1
events = pd.DataFrame(event_rows)
events.to_csv('events.csv', index=False)

# ── TABLE 4: subscriptions ────────────────────────────────────────────────
conv_users = trials[trials['converted']==1].merge(users, on='user_id')
sub_rows = []
for i, row in conv_users.iterrows():
    p = np.random.choice(plans, p=[0.5, 0.35, 0.15])
    churn = None
    if random.random() < 0.25:
        churn_d = (datetime.strptime(row['conversion_date'],'%Y-%m-%d')
                   + timedelta(days=random.randint(30,270)))
        churn = churn_d.strftime('%Y-%m-%d')
    sub_rows.append({
        'sub_id': 2001 + len(sub_rows),
        'user_id': row['user_id'],
        'plan': p, 'mrr': plan_mrr[p], 'churn_date': churn
    })
subscriptions = pd.DataFrame(sub_rows)
subscriptions.to_csv('subscriptions.csv', index=False)

# ── TABLE 5: ab_test_results ──────────────────────────────────────────────
ab_rows = []
for _, u in users.iterrows():
    row_t = trials[trials['user_id']==u['user_id']].iloc[0]
    is_trt = u['onboarding_variant'] == 'Treatment'
    onb_complete = int(random.random() < (0.62 if is_trt else 0.38))
    ab_rows.append({
        'test_id': 3001 + len(ab_rows),
        'user_id': u['user_id'],
        'group_name': u['onboarding_variant'],
        'completed_onboarding': onb_complete,
        'converted': int(row_t['converted'])
    })
ab_test = pd.DataFrame(ab_rows)
ab_test.to_csv('ab_test_results.csv', index=False)

print('✅ All 5 CSV files generated successfully!')
print(f'  users.csv          → {len(users)} rows')
print(f'  trials.csv         → {len(trials)} rows')
print(f'  events.csv         → {len(events)} rows')
print(f'  subscriptions.csv  → {len(subscriptions)} rows')
print(f'  ab_test_results.csv → {len(ab_test)} rows')
